# Phase 2 Training: DeBERTa-v3-base

Fine-tunes `microsoft/deberta-v3-base` on the cognitive distortion dataset.

**Runtime:** GPU (T4 free tier is fine, L4 is faster)
**Expected time:** 45-60 minutes on T4

Run all cells top to bottom. The model is saved to Google Drive at the end.

In [ ]:
# Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found - switch runtime to GPU')

In [ ]:
# Mount Google Drive (model will be saved here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q transformers==4.44.0 datasets torch scikit-learn pandas sentencepiece protobuf
print('Dependencies installed.')

## Setup: Upload your project files

**Option A (recommended):** Clone from GitHub
```
!git clone https://github.com/MihirApte/Cognitive-Distortion-Detector.git /content/NLP_CBT
%cd /content/NLP_CBT
```

**Option B:** Upload from Drive (if repo is already there)
```
import shutil
shutil.copytree('/content/drive/MyDrive/NLP_CBT', '/content/NLP_CBT')
%cd /content/NLP_CBT
```

Run whichever applies to you in the next cell.

In [ ]:
# Option A: Clone from GitHub (edit URL if needed)
!git clone https://github.com/MihirApte/Cognitive-Distortion-Detector.git /content/NLP_CBT
%cd /content/NLP_CBT
!ls

In [ ]:
# Verify processed data exists
import os
data_dir = '/content/NLP_CBT/data/processed'
for f in ['train.csv', 'val.csv', 'test.csv']:
    path = os.path.join(data_dir, f)
    if os.path.exists(path):
        import pandas as pd
        df = pd.read_csv(path)
        print(f'{f}: {len(df)} rows')
    else:
        print(f'MISSING: {path}')
        print('Upload processed data from Google Drive or re-run preprocessing.')

## Upload processed data

The processed CSVs are in `.gitignore` so not on GitHub.
Copy them from your local machine to Drive first, then run:

```python
import shutil, os
os.makedirs('/content/NLP_CBT/data/processed', exist_ok=True)
for f in ['train.csv', 'val.csv', 'test.csv']:
    shutil.copy(f'/content/drive/MyDrive/NLP_CBT_data/{f}',
                f'/content/NLP_CBT/data/processed/{f}')
print('Data copied.')
```

In [ ]:
# Train DeBERTa-v3-base
# This takes 45-60 min on T4. Do not close the browser tab.
import sys
sys.path.insert(0, '/content/NLP_CBT')

from src.train_phase2 import train

results = train(
    model_key='deberta',
    epochs=5,
    batch_size=8,
    use_focal_loss=True,
    global_threshold=0.40,
)

print('\nTraining complete!')
print(f'Test Macro F1: {results["macro_f1"]}')

In [ ]:
# Save model and results to Google Drive
import shutil, os

drive_model_dir = '/content/drive/MyDrive/NLP_CBT_models/deberta'
drive_results   = '/content/drive/MyDrive/NLP_CBT_results/deberta_results.json'

os.makedirs(os.path.dirname(drive_results), exist_ok=True)

# Copy model
if os.path.exists(drive_model_dir):
    shutil.rmtree(drive_model_dir)
shutil.copytree(
    '/content/NLP_CBT/models/deberta_cognitive_distortion',
    drive_model_dir
)

# Copy results JSON
shutil.copy(
    f'/content/NLP_CBT/results/deberta_results.json',
    drive_results
)

print(f'Model saved to: {drive_model_dir}')
print(f'Results saved to: {drive_results}')

In [ ]:
# Print final summary table
import json
with open('/content/NLP_CBT/results/deberta_results.json') as f:
    r = json.load(f)

print(f'Model: {r["model"]}')
print(f'Macro F1 (threshold={r["threshold"]}): {r["macro_f1"]}')
print(f'Micro F1: {r["micro_f1"]}')
print(f'Macro F1 with per-label thresholds: {r["swept_macro_f1"]}')
print()
print(f'{"Label":<42} {"F1":>6}')
print('-' * 50)
for label, f1 in r['per_label_f1'].items():
    print(f'{label:<42} {f1:.4f}')

print()
print('Copy results/{results_file} back to your local machine and run:')
print('  python -m src.generate_comparison_table')
print('to update the README comparison table.')